Extract data from csv

In [520]:
import pandas as pd
import numpy as np

In [521]:
df = pd.read_csv('data/bronze/dirty_cafe_sales.csv')

In [522]:
df.sample(5)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
5915,TXN_4321091,Smoothie,2,4.0,8.0,Cash,In-store,2023-06-15
4986,TXN_7603895,Smoothie,5,4.0,20.0,Credit Card,In-store,UNKNOWN
2802,TXN_7323833,ERROR,3,4.0,ERROR,Credit Card,NaN,2023-04-09
3004,TXN_3352256,Sandwich,2,4.0,8.0,NaN,Takeaway,2023-09-08
2173,TXN_8397155,Salad,4,5.0,20.0,Credit Card,Takeaway,2023-08-16


In [523]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [552]:
df_copy = df.copy()

In [553]:
df_copy.columns = [c.lower().replace(' ', '_') for c in df_copy.columns]

In [554]:
df_copy.columns

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date'],
      dtype='object')

In [555]:
df_copy['item'].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'UNKNOWN',
       'Sandwich', nan, 'ERROR', 'Juice', 'Tea'], dtype=object)

In [ ]:
df_copy['is_error'] = False

In [ ]:
df_copy.loc[df_copy['item'] == 'ERROR', 'is_error'] = True

In [ ]:
df_copy['is_missing'] = False

In [ ]:
df_copy.loc[(df_copy['item'] == 'UNKNOWN') | (df_copy['item'].isna()), 'is_missing'] = True

In [ ]:
df_copy.loc[(df_copy['item'] == 'UNKNOWN') | (df_copy['item'].isna()), 'is_missing'] = True

In [ ]:
df_copy.loc[(df_copy['item'] == 'UNKNOWN') | (df_copy['item'].isna()), 'is_missing'] = True

In [558]:
df_copy[df_copy['is_missing'] == True]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06,False,True
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28,False,True
30,TXN_1736287,NaN,5,2.0,10.0,Digital Wallet,NaN,2023-06-02,False,True
31,TXN_8927252,UNKNOWN,2,1.0,ERROR,Credit Card,ERROR,2023-11-06,False,True
33,TXN_7710508,UNKNOWN,5,1.0,5.0,Cash,NaN,ERROR,False,True
...,...,...,...,...,...,...,...,...,...,...
9876,TXN_3105633,NaN,1,2.0,2.0,NaN,In-store,2023-03-30,False,True
9885,TXN_4659954,NaN,3,4.0,12.0,Credit Card,In-store,NaN,False,True
9946,TXN_8807600,UNKNOWN,1,4.0,4.0,Cash,Takeaway,2023-09-24,False,True
9994,TXN_7851634,UNKNOWN,4,4.0,16.0,NaN,NaN,2023-01-08,False,True


In [559]:
df_copy.replace(['ERROR', 'UNKNOWN', 'nan'], np.nan, inplace=True)

In [560]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    10000 non-null  object
 1   item              9031 non-null   object
 2   quantity          9521 non-null   object
 3   price_per_unit    9467 non-null   object
 4   total_spent       9498 non-null   object
 5   payment_method    6822 non-null   object
 6   location          6039 non-null   object
 7   transaction_date  9540 non-null   object
 8   is_error          10000 non-null  bool  
 9   is_missing        10000 non-null  bool  
dtypes: bool(2), object(8)
memory usage: 644.7+ KB


In [561]:
df_copy['price_per_unit'] = df_copy['price_per_unit'].astype(float)

In [562]:
df_copy['total_spent'] = df_copy['total_spent'].astype(float)

In [564]:
df_copy['quantity'] = pd.to_numeric(df_copy['quantity'], errors='coerce').astype('Int64')

In [536]:
type(df_copy['quantity'][0])

numpy.int64

In [566]:
df_copy.loc[
   (df_copy['price_per_unit'].isna() &
    df_copy['total_spent'].notna() &
    df_copy['quantity'].notna() & 
    df_copy['is_error'] == False
    ), 'price_per_unit'] = df_copy['total_spent']/df_copy['quantity']

/var/folders/c2/3hrpxwbd3zlfzsl6h71y__t80000gn/T/ipykernel_94489/1923880027.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<FloatingArray>
[ 2.0,  3.0, <NA>,  5.0,  2.0,  4.0,  3.0,  4.0,  3.0,  4.0,
 ...
  2.0,  4.0,  4.0, <NA>,  4.0,  2.0,  1.0,  2.0,  1.0,  4.0]
Length: 9989, dtype: Float64' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_copy.loc[


In [568]:
df_copy['quantity'] = df_copy['quantity'].replace('<NA>', np.nan)

In [569]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    10000 non-null  object 
 1   item              9031 non-null   object 
 2   quantity          9521 non-null   Int64  
 3   price_per_unit    9028 non-null   Float64
 4   total_spent       9498 non-null   float64
 5   payment_method    6822 non-null   object 
 6   location          6039 non-null   object 
 7   transaction_date  9540 non-null   object 
 8   is_error          10000 non-null  bool   
 9   is_missing        10000 non-null  bool   
dtypes: Float64(1), Int64(1), bool(2), float64(1), object(5)
memory usage: 664.2+ KB


In [571]:
df_copy['item'].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', nan, 'Sandwich',
       'Juice', 'Tea'], dtype=object)

In [572]:
df_copy[df_copy['item'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
6,TXN_4433211,NaN,3,3.0,9.0,NaN,Takeaway,2023-10-06,False,True
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28,False,True
14,TXN_8915701,NaN,2,1.5,3.0,NaN,In-store,2023-03-21,True,False
30,TXN_1736287,NaN,5,2.0,10.0,Digital Wallet,NaN,2023-06-02,False,True
31,TXN_8927252,NaN,2,<NA>,NaN,Credit Card,NaN,2023-11-06,False,True
...,...,...,...,...,...,...,...,...,...,...
9951,TXN_4122925,NaN,4,1.0,4.0,NaN,Takeaway,2023-10-20,True,False
9958,TXN_4125474,NaN,2,5.0,10.0,Credit Card,In-store,2023-08-02,True,False
9981,TXN_4583012,NaN,5,4.0,20.0,Digital Wallet,NaN,2023-02-27,True,False
9994,TXN_7851634,NaN,4,4.0,16.0,NaN,NaN,2023-01-08,False,True


In [575]:
mapping_df = df_copy.loc[
   (df_copy['item'].notna()) & 
   (df_copy['price_per_unit'].notna()),
   ['item', 'price_per_unit']].drop_duplicates()

In [576]:
mapping_df.drop_duplicates(subset=['price_per_unit'], keep=False, inplace=True)

In [577]:
mapping = dict(zip(mapping_df['item'], mapping_df['price_per_unit']))

In [578]:
mapping

{'Coffee': np.float64(2.0),
 'Salad': np.float64(5.0),
 'Cookie': np.float64(1.0),
 'Tea': np.float64(1.5)}

In [579]:
for item, price in mapping.items():
   mask = (df_copy['item'].isna()) & (df_copy['price_per_unit'] == price) & (df_copy['is_error'] == False)
   df_copy.loc[mask, 'item'] = item

In [580]:
df_copy[df_copy['item'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
6,TXN_4433211,NaN,3,3.0,9.0,NaN,Takeaway,2023-10-06,False,True
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28,False,True
14,TXN_8915701,NaN,2,1.5,3.0,NaN,In-store,2023-03-21,True,False
31,TXN_8927252,NaN,2,<NA>,NaN,Credit Card,NaN,2023-11-06,False,True
36,TXN_6855453,NaN,4,3.0,12.0,NaN,In-store,2023-07-17,False,True
...,...,...,...,...,...,...,...,...,...,...
9946,TXN_8807600,NaN,1,4.0,4.0,Cash,Takeaway,2023-09-24,False,True
9951,TXN_4122925,NaN,4,1.0,4.0,NaN,Takeaway,2023-10-20,True,False
9958,TXN_4125474,NaN,2,5.0,10.0,Credit Card,In-store,2023-08-02,True,False
9981,TXN_4583012,NaN,5,4.0,20.0,Digital Wallet,NaN,2023-02-27,True,False


In [581]:
for item, price in mapping.items():
   mask = (df_copy['price_per_unit'].isna()) & (df_copy['item'] == item) & (df_copy['is_error'] == False)
   df_copy.loc[mask, 'price_per_unit'] = price

In [582]:
df_copy.loc[
   (df_copy['price_per_unit'].notna()) &
   (df_copy['total_spent'].isna()) &
   (df_copy['quantity'].notna()) &
   (df_copy['is_error'] == False),
    'total_spent'] = df_copy['price_per_unit'] * df_copy['quantity']

In [583]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    10000 non-null  object 
 1   item              9333 non-null   object 
 2   quantity          9521 non-null   Int64  
 3   price_per_unit    9447 non-null   Float64
 4   total_spent       9695 non-null   float64
 5   payment_method    6822 non-null   object 
 6   location          6039 non-null   object 
 7   transaction_date  9540 non-null   object 
 8   is_error          10000 non-null  bool   
 9   is_missing        10000 non-null  bool   
dtypes: Float64(1), Int64(1), bool(2), float64(1), object(5)
memory usage: 664.2+ KB


In [584]:
df_copy[df_copy['total_spent'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,is_error,is_missing
25,TXN_7958992,Smoothie,3,<NA>,NaN,NaN,NaN,2023-12-13,False,False
31,TXN_8927252,NaN,2,<NA>,NaN,Credit Card,NaN,2023-11-06,False,True
65,TXN_4987129,Sandwich,3,<NA>,NaN,NaN,In-store,2023-10-20,False,False
94,TXN_6289610,Juice,3,<NA>,NaN,Cash,Takeaway,2023-08-07,False,False
143,TXN_8495063,Juice,1,<NA>,NaN,Cash,NaN,2023-05-31,False,False
...,...,...,...,...,...,...,...,...,...,...
9890,TXN_2749289,Smoothie,2,<NA>,NaN,Digital Wallet,Takeaway,2023-05-05,False,False
9893,TXN_3809533,Juice,2,<NA>,NaN,Digital Wallet,Takeaway,2023-02-02,False,False
9977,TXN_5548914,Juice,2,<NA>,NaN,Digital Wallet,In-store,2023-11-04,False,False
9988,TXN_9594133,Cake,5,<NA>,NaN,NaN,NaN,NaN,False,False
